# Generation of transient profile data for EFCF Extended Abstract, ...

- V3: 100x higher fatigue, include also some real op val data

In [1]:
from electrolyzer import Stack, Supervisor, run_electrolyzer_zbt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm # matplotlib's color map library
import matplotlib as mpl
import electrolyzer.inputs.validation as val
import plotly.graph_objects as go; import numpy as np
from plotly_resampler import FigureResampler, FigureWidgetResampler
import pandas as pd
import math
import copy
from plotly.subplots import make_subplots
from solardatatools.dataio import get_pvdaq_data
from sklearn.preprocessing import MinMaxScaler

In [2]:
def overwrite_protocol_with_performance_points(protocol,
                                               performance_point_interval,
                                               performance_point_holding,
                                              performance_point_Prel=0.8):
    """
    Function which overwrites timesteps of a profile with specific performance points
    :param protocol:
    :return:
    """
    overall_time = protocol.ts.iloc[-1]
    pp_start_ts = np.arange(0, overall_time, performance_point_interval)
    pp_holding = [i for i in range(performance_point_holding)]
    pp_ts = [[start_ts + hold for hold in pp_holding] for start_ts in pp_start_ts]
    pp_ts = [x for xs in pp_ts
             for x in xs]

    protocol.loc[protocol.ts.isin(pp_ts), "power"] = P_min+ performance_point_Prel *  (P_max-P_min)

    return protocol

## Stack definition & initialization (Required to extract power of the system)

In [3]:
# Initialize system
fname_input_modeling = "./DEl_PM4_modeling_options.yaml"
modeling_options = val.load_modeling_yaml(fname_input_modeling)
elec_sys = Supervisor.from_dict(modeling_options["electrolyzer"])
elec = elec_sys.stacks[0]

In [4]:
modeling_options

{'general': {'verbose': False},
 'electrolyzer': {'dt': 1,
  'stack': {'cell_area': 1000.0,
   'max_current': 2000,
   'temperature': 60,
   'n_cells': 100,
   'include_degradation_penalty': True,
   'dt': 1},
  'control': {'n_stacks': 1, 'control_type': 'BaselineDeg'},
  'name': 'electrolyzer_001',
  'description': 'A PEM electrolyzer model',
  'initialize': False,
  'initial_power_kW': 0.0,
  'costs': {}}}

## Long Term Load Profiles

### Combined Cycles
Each profile is one year / 8766hrs long

In [5]:
powersignals = {"RES_Wind":[],
               "RES_Solar":[],
               "Constant_High":[],
               "Constant_Mid":[]}
P_min= elec.min_power
P_max = elec.stack_rating
print(P_min,P_max)

46660.87847588875 466608.7847588875


In [6]:
stack_data = elec.create_polarization_data(current_interval=1,
                                          temp_min=40,
                                          temp_max=80,
                                          temp_interval=2)
stack_data.loc[(stack_data.power_kW > P_min/1000 *0.99) & (stack_data.power_kW < P_min/1000 *1.01),:]

,current_A,power_kW,voltage_V,currentdens_Acm-2,cellvoltage_V,temp_C
262,262,46.198536,176.330288,0.262,1.763303,40
263,263,46.389881,176.387381,0.263,1.763874,40
264,264,46.581315,176.444376,0.264,1.764444,40
265,265,46.772838,176.501275,0.265,1.765013,40
266,266,46.964448,176.558077,0.266,1.765581,40
...,...,...,...,...,...,...
261,261,46.349192,177.583109,0.261,1.775831,80
262,262,46.539421,177.631380,0.262,1.776314,80
263,263,46.729719,177.679539,0.263,1.776795,80
264,264,46.920083,177.727589,0.264,1.777276,80


### RES 

In [7]:
#For today’s example, we’re loading data from NREL’s PVDAQ API, which is a publically available PV generatation data set.
data_real = get_pvdaq_data(sysid=34, year=range(2011, 2012), api_key='DEMO_KEY')


[============================================================] 100.0% ...queries complete in 2.5 seconds       



In [8]:
data_real["Time"] = data_real.index
data_real["Time_seconds"] = data_real["Time"] -  data_real["Time"].iloc[0]
data_real["Time_seconds"]= data_real["Time_seconds"].dt.total_seconds()

In [9]:
data_real

,SiteID,ac_current,ac_power,ac_voltage,ambient_temp,dc_current,dc_power,dc_voltage,inverter_error_code,inverter_temp,module_temp,poa_irradiance,power_factor,relative_humidity,wind_direction,wind_speed,Time,Time_seconds
2011-01-01 00:00:00,34.0,0.0,-200.0,284.0,-3.353332,0.0,-200.0,16.0,0.0,37.0,-7.105555,0.0,0.0,53.513,315.270,0.483250,2011-01-01 00:00:00,0.0
2011-01-01 00:15:00,34.0,0.0,-300.0,284.0,-3.381110,0.0,-200.0,16.0,0.0,36.0,-6.944444,0.0,0.0,53.581,308.835,0.698724,2011-01-01 00:15:00,900.0
2011-01-01 00:30:00,34.0,0.0,-300.0,284.0,-3.257777,0.0,-200.0,16.0,0.0,36.0,-6.344444,0.0,0.0,53.413,272.678,0.218156,2011-01-01 00:30:00,1800.0
2011-01-01 00:45:00,34.0,0.0,-200.0,283.0,-3.296666,0.0,0.0,15.0,0.0,36.0,-6.655555,0.0,0.0,52.406,55.913,0.159146,2011-01-01 00:45:00,2700.0
2011-01-01 01:00:00,34.0,0.0,-300.0,284.0,-3.426110,0.0,-200.0,14.0,0.0,35.0,-7.405555,0.0,0.0,53.588,152.145,0.240508,2011-01-01 01:00:00,3600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011-12-31 22:45:00,34.0,0.0,-300.0,286.0,6.068891,0.0,0.0,5.0,0.0,40.0,2.188890,0.0,0.0,38.975,302.422,1.586992,2011-12-31 22:45:00,31531500.0
2011-12-31 23:00:00,34.0,0.0,-200.0,286.0,5.770557,0.0,0.0,4.0,0.0,39.0,2.883335,0.0,0.0,37.663,304.088,1.581180,2011-12-31 23:00:00,31532400.0
2011-12-31 23:15:00,34.0,0.0,-200.0,286.0,5.466669,0.0,0.0,4.0,0.0,39.0,2.383335,0.0,0.0,37.063,304.470,1.439022,2011-12-31 23:15:00,31533300.0
2011-12-31 23:30:00,34.0,0.0,-200.0,285.0,5.135002,0.0,0.0,4.0,0.0,39.0,2.127779,0.0,0.0,38.138,304.898,1.369284,2011-12-31 23:30:00,31534200.0


In [10]:
# Interpolation to 1s time step
time_s_new = np.arange(365*24*3600)
wind_speed_new = np.interp(time_s_new, data_real["Time_seconds"], data_real["wind_speed"])
solar_new = np.interp(time_s_new, data_real["Time_seconds"], data_real["ac_power"])

In [11]:
data_real_interpolated = pd.DataFrame({"Time_seconds":time_s_new,
                                      "wind_speed":wind_speed_new,
                                      "ac_power":solar_new})

In [12]:
data_real_interpolated

,Time_seconds,wind_speed,ac_power
0,0,0.483250,-200.000000
1,1,0.483490,-200.111111
2,2,0.483729,-200.222222
3,3,0.483968,-200.333333
4,4,0.484208,-200.444444
...,...,...,...
31535995,31535995,1.008522,-200.000000
31535996,31535996,1.008522,-200.000000
31535997,31535997,1.008522,-200.000000
31535998,31535998,1.008522,-200.000000


In [13]:
# Scale wind and power measurements between p_min and p_max
# Original
wind_scaler = MinMaxScaler(feature_range=(P_min, P_max))
wind_scaler.fit(data_real[["wind_speed"]])
                
solar_scaler = MinMaxScaler(feature_range=(P_min, P_max))
solar_scaler.fit(data_real[["ac_power"]])

data_real["wind_speed_scaled"] = wind_scaler.transform(data_real[["wind_speed"]])
data_real["solar_scaled"] = solar_scaler.transform(data_real[["ac_power"]])
data_real_interpolated["wind_speed_scaled"] = wind_scaler.transform(data_real_interpolated[["wind_speed"]])
data_real_interpolated["solar_scaled"] = solar_scaler.transform(data_real_interpolated[["ac_power"]])

# Adjustment:
data_real_interpolated["wind_speed_scaled"] = data_real_interpolated["wind_speed_scaled"]* 2
data_real_interpolated["wind_speed_scaled"] = data_real_interpolated["wind_speed_scaled"].clip(upper=P_max)

# data_real_interpolated["solar_scaled"] = data_real_interpolated["solar_scaled"]* 2
data_real_interpolated["solar_scaled"] = data_real_interpolated["solar_scaled"].clip(upper=P_max, lower=P_min*1.2)

In [14]:
fig = FigureResampler(go.Figure())

fig = FigureResampler(
    make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=["Wind","Power"],
        #vertical_spacing=0.05,
    )
)

fig.add_trace(go.Scattergl(name=f'Wind', showlegend=True), hf_x=data_real.Time_seconds, hf_y=data_real.wind_speed_scaled,row=1,col=1)
fig.add_trace(go.Scattergl(name=f'Power', showlegend=True), hf_x=data_real.Time_seconds, hf_y=data_real.solar_scaled,row=2,col=1)

fig.add_trace(go.Scattergl(name=f'Wind adjusted', showlegend=True), hf_x=data_real_interpolated.Time_seconds, hf_y=data_real_interpolated.wind_speed_scaled,row=1,col=1)
fig.add_trace(go.Scattergl(name=f'Power adjusted', showlegend=True), hf_x=data_real_interpolated.Time_seconds, hf_y=data_real_interpolated.solar_scaled,row=2,col=1)
    # fig.add_trace(go.Scattergl(name=f'{n}', showlegend=True), hf_x=df.time_h, hf_y=df["stack_1_deg"],row=2,col=1)

fig.update_layout( autosize=False, width=1000, height=800,
                  xaxis_title="Time",
                  yaxis_title="Power",
                 xaxis2_title="Time",
                  yaxis2_title="Power",
                 )

fig.show_dash(mode='inline', port=8097)

In [15]:
# Combine 


In [16]:
powersignals["RES_Wind"] = data_real_interpolated.wind_speed_scaled 
powersignals["RES_Solar"] = data_real_interpolated.solar_scaled 

# powersignals = {"RES_Wind":[],
#                "RES_Solar":[],
#                "Constant_High":[],
#                "Constant_Mid":[]}

## Pretty Plotting without performance point

In [17]:
# Plot combined cycle signal

powersignal = powersignals["RES_Wind"]
fig = FigureResampler(go.Figure())

fig.add_trace(go.Scattergl(name=f'Wind', showlegend=True,
                          line=dict(color='black',# color='#808080',
                                   # width=.5)),
                                    width=0.5),
                          mode='lines'),
              hf_x=[x/3600  for x in list(range(len(powersignal)))],
              hf_y=[x/P_max for x in powersignal],
max_n_samples=50000)
    # fig.add_trace(go.Scattergl(name=f'{n}', showlegend=True), hf_x=df.time_h, hf_y=df["stack_1_deg"],row=2,col=1)

fig.update_layout( autosize=False, 
                  width=600, height=400,
                  template="simple_white",
                 xaxis_title="Time [h]",
                 yaxis_title="Load [%]")

fig.show_dash(mode='inline', port=5088)

In [18]:
### Constant Operation
powersignals["Constant_High"] = [P_max * 0.9] * len(powersignals["RES_Wind"])
powersignals["Constant_Mid"] = [P_max * 0.5] * len(powersignals["RES_Wind"])

In [19]:
pd.to_pickle(powersignals, "signals_no_pp_v5.pkl")

In [20]:
powersignals=pd.read_pickle("signals_no_pp_v5.pkl")


In [21]:
powersignals_performance_point = copy.deepcopy(powersignals)

In [22]:
powersignals_pp=dict()
for n,ps in powersignals_performance_point.items():
    print(ps[:10])
    power_df= pd.DataFrame({"power":ps})
    print(power_df.head())
    power_df.loc[:,"ts"] = power_df.index
    power_df = overwrite_protocol_with_performance_points(power_df, 60*60*3,10)
    powersignals_pp[n] = list(power_df.power)
powersignals_performance_point = powersignals_pp

0    136363.729711
1    136385.053825
2    136406.377940
3    136427.702054
4    136449.026169
5    136470.350283
6    136491.674397
7    136512.998512
8    136534.322626
9    136555.646740
Name: wind_speed_scaled, dtype: float64
           power
0  136363.729711
1  136385.053825
2  136406.377940
3  136427.702054
4  136449.026169
0    55993.054171
1    55993.054171
2    55993.054171
3    55993.054171
4    55993.054171
5    55993.054171
6    55993.054171
7    55993.054171
8    55993.054171
9    55993.054171
Name: solar_scaled, dtype: float64
          power
0  55993.054171
1  55993.054171
2  55993.054171
3  55993.054171
4  55993.054171
[np.float64(419947.90628299874), np.float64(419947.90628299874), np.float64(419947.90628299874), np.float64(419947.90628299874), np.float64(419947.90628299874), np.float64(419947.90628299874), np.float64(419947.90628299874), np.float64(419947.90628299874), np.float64(419947.90628299874), np.float64(419947.90628299874)]
           power
0  419947.906283
1 

In [23]:
# Plot combined cycle signal
fig = FigureResampler(go.Figure())

for n,ps in powersignals_performance_point.items():
    fig.add_trace(go.Scattergl(name=n, showlegend=True), hf_x=list(range(len(ps))), hf_y=ps)
        # fig.add_trace(go.Scattergl(name=f'{n}', showlegend=True), hf_x=df.time_h, hf_y=df["stack_1_deg"],row=2,col=1)

fig.update_layout( autosize=False, width=1000, height=400,
                 xaxis_title="Time")

fig.show_dash(mode='inline', port=5081)

In [24]:
pd.to_pickle(powersignals_performance_point, "signals_pp_v5.pkl")

## AST Cycle Generation

In [24]:
ast_signals =dict()

In [26]:
# doe_ast = pd.read_pickle("./doe_dn_deno_v2.pkl")

np.float64(46660.87847588875)

In [28]:

# ast_signals["doe"]= [i_rel * (P_max-(1.2* P_min))+(1.2*P_min) for i_rel in doe_ast.i_rel] + \
#  [i_rel * (P_max-(1.2* P_min))+(1.2*P_min) for i_rel in doe_ast.i_rel]
# ast_signals["doe"] = ast_signals["doe"][:3600*1000]

In [29]:
# # Plot combined cycle signal
# fig = FigureResampler(go.Figure())

# for n,ps in ast_signals.items():
#     fig.add_trace(go.Scattergl(name=n, showlegend=True), hf_x=list(range(len(ps))), hf_y=ps)
#         # fig.add_trace(go.Scattergl(name=f'{n}', showlegend=True), hf_x=df.time_h, hf_y=df["stack_1_deg"],row=2,col=1)

# fig.update_layout( autosize=False, width=1000, height=400,
#                  xaxis_title="Time")

# fig.show_dash(mode='inline', port=5086)

In [25]:
#for period_minute in [4/60,12/6,1]:
for name,period_minute in zip(["24s","48s"],[24/60,48/60]):
    amplitude =  0.5 * (P_max-(1.2*P_min))
    y_offset = amplitude + (1.2*P_min)
    ast_signals[f"Cyclic_{name}"]= [ y_offset +  amplitude * math.sin(t/(period_minute * 60) * 2 * math.pi)  for t in range(1000 * 3600)]

In [26]:
# Plot combined cycle signal
fig = FigureResampler(go.Figure())

for n,ps in ast_signals.items():
    fig.add_trace(go.Scattergl(name=n, showlegend=True), hf_x=list(range(len(ps))), hf_y=ps)
        # fig.add_trace(go.Scattergl(name=f'{n}', showlegend=True), hf_x=df.time_h, hf_y=df["stack_1_deg"],row=2,col=1)

fig.update_layout( autosize=False, width=1000, height=400,
                 xaxis_title="Time")

fig.show_dash(mode='inline', port=5086)

### Pretty Plot without performance points

In [27]:
# Plot combined cycle signal

# powersignal = powersignals["RES_Wind"]
fig = FigureResampler(go.Figure())

for n,ps in ast_signals.items():
    fig.add_trace(go.Scattergl(name=n, showlegend=True,
                              line=dict(color='black',# color='#808080',
                                       # width=.5)),
                                        width=1),
                              mode='lines'),
                  hf_x=[x/3600  for x in list(range(len(ps)))],
                  hf_y=[x/P_max * 100 for x in ps],
    max_n_samples=50000)
    # fig.add_trace(go.Scattergl(name=f'{n}', showlegend=True), hf_x=df.time_h, hf_y=df["stack_1_deg"],row=2,col=1)

fig.update_layout( autosize=False, 
                  width=600, height=400,
                  template="simple_white",
                 xaxis_title="Time [h]",
                 yaxis_title="Load [%]")

fig.show_dash(mode='inline', port=5077)

In [28]:
ast_signals_performance_point = copy.deepcopy(ast_signals)

ast_signals_pp=dict()
for n,ps in ast_signals_performance_point.items():
    print(ps[:10])
    power_df= pd.DataFrame({"power":ps})
    print(power_df.head())
    power_df.loc[:,"ts"] = power_df.index
    power_df = overwrite_protocol_with_performance_points(power_df, 60*60*3,10)
    ast_signals_pp[n] = list(power_df.power)
ast_signals_performance_point = ast_signals_pp

[np.float64(261300.919464977), np.float64(314438.50511238386), np.float64(363954.85211193224), np.float64(406475.5032452353), np.float64(439102.7464062569), np.float64(459613.0888926422), np.float64(466608.7847588875), np.float64(459613.0888926422), np.float64(439102.746406257), np.float64(406475.5032452353)]
           power
0  261300.919465
1  314438.505112
2  363954.852112
3  406475.503245
4  439102.746406
[np.float64(261300.919464977), np.float64(288098.9733546184), np.float64(314438.50511238386), np.float64(339868.8380472002), np.float64(363954.85211193224), np.float64(386284.42892802786), np.float64(406475.5032452353), np.float64(424182.60018396383), np.float64(439102.7464062569), np.float64(450980.65407360526)]
           power
0  261300.919465
1  288098.973355
2  314438.505112
3  339868.838047
4  363954.852112


In [ ]:
# Plot combined cycle signal
fig = FigureResampler(go.Figure())

for n,ps in ast_signals_performance_point.items():
    fig.add_trace(go.Scattergl(name=n, showlegend=True), hf_x=list(range(len(ps))), hf_y=ps)
        # fig.add_trace(go.Scattergl(name=f'{n}', showlegend=True), hf_x=df.time_h, hf_y=df["stack_1_deg"],row=2,col=1)

fig.update_layout( autosize=False, width=1000, height=400,
                 xaxis_title="Time")

fig.show_dash(mode='inline', port=5086)

In [ ]:
pd.to_pickle(ast_signals_performance_point, "ast_signals_pp_v5.pkl")